# TaaS Studio — Reviewer Quickstart (Colab)

**What this is:** a clean-room launch of the TaaS Studio control plane on a free Colab runtime — the same pinned build a reviewer should evaluate. Run every cell top-to-bottom (~5 minutes). You will get a live Studio URL and a guided tour of the agent/tool lifecycle surfaces.

**What you are looking at:** TaaS Studio is a *provenance-aware control plane*. It organizes work around the AI agent/tool lifecycle — **DEFINE → BUILD → DISCOVER → VERIFY → OPERATE → EVALUATE & MANAGE**. It never invokes MCP tools; discovery inspects metadata, assurance inspects evidence, and the audit ledger is tamper-evident. *Proposal ≠ evidence ≠ authorization ≠ execution.*

**After this notebook:** open the printed Studio link, paste the printed token, then follow the tour in the markdown cells. For the full contract acceptance suite, run `notebooks/colab/taas_studio_acceptance_colab.ipynb` instead (or after).

## 1 · Checkout the pinned revision

In [ ]:
import os, sys, subprocess, pathlib, time, shutil, urllib.request, secrets

from IPython.display import display, HTML



REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"

TAAS_COMMIT = "ad63b7e455fe23b8cb70f0ff0110e0b9bb7b3e7e"

WORKDIR = pathlib.Path("/content/taas-provenance-discovery")

APP_BIND_HOST, APP_HEALTH_HOST, APP_PORT = "0.0.0.0", "127.0.0.1", 8000



if not WORKDIR.exists():

    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)

subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)

subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)

actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()

assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)

assert (WORKDIR / "pyproject.toml").is_file()

assert (WORKDIR / "src" / "taas").is_dir()

print("TaaS revision:", actual)

print("TaaS API source:", WORKDIR / "src" / "taas" / "studio" / "api.py")

## 2 · Hardware check (informational)

In [ ]:
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
GPU_AVAILABLE = gpu_check.returncode == 0
print(gpu_check.stdout if GPU_AVAILABLE else "No NVIDIA GPU visible; Hugging Face CPU and Ollama remain available.")
print("Python:", sys.version.split()[0])

## 3 · Install TaaS Studio

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[test,studio,hf]"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(install.stdout[-4000:])
if install.returncode:
    print(install.stderr[-4000:])
    raise RuntimeError("TaaS Studio installation failed")
import_check = subprocess.run(
    [
        sys.executable, "-c",
        "import traceback, importlib; "
        "importlib.import_module('taas'); "
        "importlib.import_module('fastapi'); "
        "importlib.import_module('uvicorn'); "
        "importlib.import_module('torch'); "
        "importlib.import_module('transformers'); "
        "importlib.import_module('safetensors'); "
        "importlib.import_module('sentencepiece'); "
        "importlib.import_module('tiktoken'); "
        "importlib.import_module('taas.studio.ollama_runtime'); "
        "importlib.import_module('taas.studio.api'); "
        "print('TaaS Model/Agent Studio imports OK')"
    ],
    cwd=WORKDIR, text=True, capture_output=True
)
print(import_check.stdout)
if import_check.returncode:
    print("=== IMPORT CHECK STDERR / TRACEBACK ===")
    print(import_check.stderr[-12000:])
    print("Python executable:", sys.executable)
    print("Working directory:", WORKDIR)
    print("Package source exists:", (WORKDIR / "src" / "taas" / "__init__.py").is_file())
    raise RuntimeError("TaaS Model/Agent Studio import check failed; see traceback above")

## 4 · Demo MCP server (inspection-only fixture)

A tiny real MCP stdio server so the *MCP connections* surface has something genuine to inspect. Studio runs `tools/list` only — it never calls tools.

In [ ]:
import json

# mcp client+server SDK is an optional extra — install it only for the demo server.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mcp>=1.9,<2"],
    check=True)  # pinned <2: FastMCP API (mcp 2.x renamed it)

# Optional-but-default demo MCP server: gives DISCOVER/MCP surfaces a real
# stdio server to inspect (tools/list only — Studio never invokes tools).
demo_server = pathlib.Path("/content/taas-demo-mcp-server.py")
demo_server.write_text('''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("taas-demo-tools")


@mcp.tool()
def echo_message(text: str) -> str:
    """Return the input text unchanged."""
    return text


@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b


if __name__ == "__main__":
    mcp.run()
''')
mcp_config = pathlib.Path("/content/taas-mcp-servers.json")
mcp_config.write_text(json.dumps([{
    "id": "taas-demo",
    "name": "TaaS Demo MCP Server",
    "command": sys.executable,
    "args": [str(demo_server)],
    "env": [],
}]))
print("Demo MCP server script:", demo_server)
print("MCP registry config:", mcp_config)

## 5 · Launch Studio

Generates an ephemeral API token, persists all stores under `/content/taas-data`, configures the demo MCP server, and starts the API. Copy the token — it is shown once.

In [ ]:
api_token = secrets.token_urlsafe(32)
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ.pop("TAAS_MODEL_PROFILE", None)
_data_dir = pathlib.Path("/content/taas-data")
_data_dir.mkdir(parents=True, exist_ok=True)
# Persist the evidence attestation key so re-launching Studio does not
# invalidate signatures/approvals recorded earlier in the session.
_evidence_key_file = _data_dir / "evidence-attestation.key"
if _evidence_key_file.is_file():
    os.environ["TAAS_EVIDENCE_ATTESTATION_KEY"] = _evidence_key_file.read_text().strip()
else:
    os.environ["TAAS_EVIDENCE_ATTESTATION_KEY"] = secrets.token_hex(32)
    _evidence_key_file.write_text(os.environ["TAAS_EVIDENCE_ATTESTATION_KEY"])
for _var, _name in {
    "TAAS_AUDIT_LEDGER_PATH": "audit.sqlite3",
    "TAAS_AGENT_REGISTRY_PATH": "agents.sqlite3",
    "TAAS_SKILL_REGISTRY_PATH": "skills.sqlite3",
    "TAAS_WORKSPACES_PATH": "workspaces.sqlite3",
    "TAAS_DISCOVERY_RUNS_PATH": "runs.sqlite3",
    "TAAS_EVIDENCE_DB_PATH": "evidence.sqlite3",
    "TAAS_API_KEYS_PATH": "api_keys.sqlite3",
}.items():
    os.environ[_var] = str(_data_dir / _name)
os.environ.setdefault("TAAS_STUDIO_ACTOR", "colab-reviewer")
if pathlib.Path("/content/taas-mcp-servers.json").is_file():
    os.environ["TAAS_MCP_SERVERS_FILE"] = "/content/taas-mcp-servers.json"
print("Temporary Studio API token (copy into Studio; do not share):")
print(api_token)

# Make rerunning this cell safe, including after the Stop cell was used.
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try:
        studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.close()

studio_log = open("/content/taas-studio.log", "a", buffering=1)
studio_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "taas.studio.api:app",
     "--host", APP_BIND_HOST, "--port", str(APP_PORT)],
    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT
)

def wait_http(url, attempts=40):
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError("Studio exited; inspect /content/taas-studio.log")
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.status, response.read().decode("utf-8")
        except Exception as exc:
            last = exc
            time.sleep(1)
    raise RuntimeError(f"Studio did not become ready: {last}")

print("Health:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/health"))
print("Readiness:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/ready"))
print("Studio process PID:", studio_process.pid)


## 6 · Open Studio

In [ ]:
from google.colab import output

# Never reuse a proxy URL cached by an earlier server/runtime lifecycle.
if "studio_process" not in globals() or studio_process.poll() is not None:
    raise RuntimeError("TaaS Studio is not running. Execute the Launch Studio cell and inspect /content/taas-studio.log.")
status, health_body = wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/health")
if status != 200:
    raise RuntimeError(f"TaaS Studio health check failed: HTTP {status}: {health_body}")
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
if not proxy_url or not proxy_url.startswith("https://"):
    raise RuntimeError(f"Colab did not return a valid HTTPS proxy URL: {proxy_url!r}")
# Add a per-launch cache key so an old Colab proxy response cannot be reused.
proxy_separator = "&" if "?" in proxy_url else "?"
fresh_proxy_url = proxy_url + proxy_separator + "studio_session=" + str(time.time_ns())
display(HTML(f'<p><a href="{fresh_proxy_url}" target="_blank" rel="noopener">Open TaaS Studio</a></p>'))
print("Studio health: HTTP", status)
print("Studio PID:", studio_process.pid)
print("Fresh Colab proxy URL generated (cache-busted) for this running server.")


## 7 · Guided tour (~10 minutes)

Open the Studio link from the cell above, paste the token, click **Connect**. Then walk the lifecycle:

| Stop | Nav section | What to do | What it proves |
|---|---|---|---|
| 1 | **Overview** | Read environment card | Live runtime diagnostics, honest GPU state |
| 2 | **MCP connections** (DISCOVER) | *Refresh configured servers* → *Inspect server tools* | Real `tools/list` on the demo server — notice *"No tools/call request was made"* |
| 3 | **Tool registry** (DISCOVER) | Browse/search tools + facets | 500+ real registry entries, metadata only |
| 4 | **Tool discovery** (DISCOVER) | Run a discovery on a GoalSpec | Persisted run, `discovery_run_id`, no execution |
| 5 | **Assurance center** (VERIFY) | Inspect evidence semantics | Evidence ≠ authorization ≠ execution, labeled inline |
| 6 | **Audit timeline** (OPERATE) | *Load audit records* → *Verify hash chain* | Tamper-evident ledger, `chain_valid`, request-id correlation |
| 7 | **Agent/Skill lifecycle** (BUILD) | Create → validate → publish → sign → revoke | Full provenance chain with signatures |
| 8 | **Platform overview** (EVALUATE & MANAGE) | Cross-check counts vs. source endpoints | Counts come from real stores; unsupported KPIs are explicit |

Every panel states its own boundary. If something is unavailable, the UI says `UNAVAILABLE` — it never fabricates.

## 8 · Optional: API sanity check

Verifies the same contract the acceptance notebook asserts — run this if the UI looks wrong.

In [ ]:
API_BASE = f"http://{APP_HEALTH_HOST}:{APP_PORT}"
req = urllib.request.Request(API_BASE + "/api/v1/platform/overview",
                             headers={"Authorization": "Bearer " + api_token})
with urllib.request.urlopen(req, timeout=10) as r:
    import json as _j
    body = _j.loads(r.read())
    print("overview HTTP", r.status, "| schema:", body.get("schema"))
    print("auth keys present:", "authentication" in body)

## 9 · Full acceptance suite

For the complete A1→A6 contract verification with the PASS/FAIL matrix and downloadable JSON report, open `notebooks/colab/taas_studio_acceptance_colab.ipynb` in a new tab and run all cells (~10 min).

## 10 · Stop Studio

In [ ]:
# if "studio_process" in globals() and studio_process.poll() is None:
#     studio_process.terminate()
#     try:
#         studio_process.wait(timeout=10)
#     except subprocess.TimeoutExpired:
#         studio_process.kill()
#         studio_process.wait(timeout=5)
# if "studio_log" in globals() and not studio_log.closed:
#     studio_log.flush()
#     studio_log.close()
# if "tunnel_process" in globals() and tunnel_process.poll() is None:
#     tunnel_process.terminate()
#     try:
#         tunnel_process.wait(timeout=10)
#     except subprocess.TimeoutExpired:
#         tunnel_process.kill()
#         tunnel_process.wait(timeout=5)
# if "tunnel_log" in globals() and not tunnel_log.closed:
#     tunnel_log.flush()
#     tunnel_log.close()
# if "ollama_process" in globals() and ollama_process.poll() is None:
#     ollama_process.terminate()
#     try:
#         ollama_process.wait(timeout=10)
#     except subprocess.TimeoutExpired:
#         ollama_process.kill()
#         ollama_process.wait(timeout=5)
# print("TaaS Studio, optional tunnel and optional Ollama daemon stopped.")
